# Where was this basalt made?

**EPS 88 · PyEarth.** Open your own copy on DataHub: [click here](https://datahub.berkeley.edu/hub/user-redirect/git-pull?repo=https%3A//github.com/AI4EPS/EPS88_PyEarth&branch=main&urlpath=lab/tree/EPS88_PyEarth/docs/notebooks/10_where_basalt.ipynb).

Basalt is the commonest volcanic rock on Earth: the crust under every ocean is made of it.
It is made in three different places, where the mantle melts, and a geologist often holds a piece without knowing which.
California's Coast Ranges, for example, hold slices of old sea floor lifted onto the continent,
and whether that crust was made at a mid-ocean ridge or above a sinking plate decides what story
the rocks tell. The rock cannot say where it came from. Its chemistry can.

Today you work with 756 basalts whose birthplace is known, compiled by the geologist
Pieter Vermeesch (2006) from published analyses. You will build each of today's methods by hand
first, with a loop you write, and only then call the library and check that it lands on your
answer: a rule with two cuts, then a decision tree; a test cut five ways, then cross-validation; a
vote of trees you grew yourself, then a random forest; columns scaled by the formula, then a support
vector machine. Nearly a third of the rocks are locked away at
the start and opened once, at the end. At home you ask whether the winner works on rocks from
somewhere else.

Every place you write something opens with a pencil icon and the words *Your turn*, and is
followed by a cell that already has the shape of the answer written in, with `...` wherever your
code goes. Replace every `...`, run every cell so your answers and figures are
saved in the file, then **download this notebook itself — the `.ipynb` file — and upload it
to Gradescope.** Not a PDF: the marking reads your notebook, and a PDF cannot be read.
In JupyterLab: **File ▸ Download**, or right-click the file in the left-hand panel and choose
**Download**.

Two habits from the first minute. A cell runs when you press **Shift+Enter**, and the notebook
remembers everything it has already run — so when something breaks and you cannot see why,
**Kernel → Restart Kernel and Run All Cells** throws the memory away and rebuilds it from the
top. That is never the wrong thing to do.

## What you'll be able to do

**The science.** Name the three places basalt melts and why, and say which element tells them
apart best.

**The code.** Last week's `LogisticRegression` and `confusion_matrix`, now with three settings ·
a grid search over two cuts, then `DecisionTreeClassifier` · cross-validation as a loop, then
`cross_val_score` · a vote of bootstrap trees, then `RandomForestClassifier` · scaling by its
formula, then `StandardScaler` and `SVC` · the best model picked with `np.argmax`, then tested once.

The four questions of the class:

1. Can two elements tell where a basalt was made, by a line or by a rule?
2. Does the score depend on which rocks we hold out?
3. Do a forest of trees, or a support vector machine, do better?
4. Which model wins when the sealed rocks are opened?

## Setup

Run the next cell once. It reads the basalts, stored with the course: one row per rock, its
setting and its chemistry. Two columns are renamed so every name follows one pattern, and one
impossible value is blanked (a rock recorded with 19 percent MnO; basalt holds well under 1
percent).

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, cross_val_score, PredefinedSplit
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import pandas as pd
import matplotlib.pyplot as plt

# house style, set once, so every plot cell below holds only what matters
plt.rcParams.update({"figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True})

CACHE = "https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data"

# One row per rock: its setting (IAB, MORB or OIB) and up to 51 chemical measurements.
# Source: Vermeesch (2006), Geochemistry, Geophysics, Geosystems 7, doi:10.1029/2005GC001092.
rocks = pd.read_csv(CACHE + "/week10_vermeesch_basalts.csv")
rocks = rocks.rename(columns={"affinity": "setting", "P2O5(wt%)": "P2O5_wt_percent"})
rocks.loc[rocks["MnO_wt_percent"] > 5, "MnO_wt_percent"] = np.nan   # one impossible value, blanked
print(len(rocks), "rocks,", rocks.shape[1] - 1, "measurements each (many blank)")

## 1. Can two elements tell where a basalt was made, by a line or by a rule?

**A rock with no address.** California's Coast Ranges hold slices of old sea floor, scraped off a
sinking plate and lifted onto the continent. Was that crust made at a mid-ocean ridge, above a
sinking plate, or on an ocean island? The answer decides the history the rocks tell, and the rock
itself cannot say. Its chemistry can.

**Basalt has three main homes in the ocean.** Basalt is made in three places, where the mantle melts: where two plates pull apart (a mid-ocean ridge), where one plate sinks under another (an island arc), and over a hot plume, usually far from any plate edge (an ocean island). (It also forms on
the continents, where they rift apart or flood with lava; today's rocks come from the three ocean homes.)

<img src="https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data/images/week10/plate_boundaries_cross_section.png" width="720">

*Among its features: an island arc above a sinking plate, a shield volcano over a hot spot, and a
mid-ocean ridge where two plates pull apart. Cross-section by José F. Vigil, USGS (public
domain), via Wikimedia Commons.*

**Each home melts the mantle in its own way, and the melt keeps the mark.** Two elements carry two of
the clearest marks. Both prefer the melt to the solid mantle, so when only a little of the mantle melts,
what there is of them gathers into a little liquid and the basalt is rich in them.

| Home | Why the mantle melts | How much melts | Titanium | Potassium | Examples |
|---|---|---|---|---|---|
| Mid-ocean ridge (**MORB**) | the plates pull apart; hot mantle rises to fill the gap and melts as the pressure on it drops | a large share | middle: diluted in much melt | low: this mantle lost it in melts long ago | Mid-Atlantic Ridge, East Pacific Rise |
| Island arc (**IAB**) | water from the sinking plate lowers the mantle's melting point, as salt melts ice | varies | low: this mantle has often melted before, and the water brings little | raised: the water carries it, with strontium, out of the sinking plate; some arcs (Tonga, Izu-Bonin) still hold little | Mariana, Izu-Bonin, Tonga, Aleutians |
| Ocean island (**OIB**) | a plume of unusually hot mantle rises, usually far from any plate edge; the thick plate above stops it rising far | a small share, deep down, of a mantle richer in both elements | high | highest | Hawaii, Samoa, the Canary Islands |

**So, a prediction:** arc basalt low in titanium and higher in potassium than ridge basalt; ridge
basalt middling in titanium and low in potassium; ocean-island basalt high in both.

**From the outside the three look alike:** dark fine-grained rock, often in rounded "pillows" where lava met cold sea water:

<img src="https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data/images/week10/pillow_basalt.jpg" width="420">

*Pillow basalt on the sea floor. Photo: NOAA (public domain), via Wikimedia Commons.*

**So geologists measure what is inside.** A rock analysis gives the common elements as oxides, in percent of the rock's weight (SiO2, TiO2), and the rare ones in parts per million (V, Sr).

**How geologists have done it: lines drawn by hand.** In 1982 the geologist John Shervais plotted
titanium against vanadium and drew lines between the settings. Two of them, at Ti/V = 20 and
Ti/V = 50, part the three: arc rocks below 20 (above the steeper line, for arc basalts hold little titanium for their vanadium),
ridge rocks between, ocean islands above 50. (That last field is for the *alkaline* ocean-island
basalts; island tholeiites, like most of Hawaii's lava, plot between 20 and 50 with the ridges. The
homework meets them.) Titanium is reported as TiO2 in weight percent; titanium
is 47.87 of TiO2's 79.87 units of mass (their atomic masses), so 1 percent TiO2 holds
10,000 × 47.87 / 79.87 ≈ 5993 parts per million of titanium. The next cell draws the
514 rocks with both elements and his two lines, and scores his rule.

In [ ]:
rocks_tv = rocks.dropna(subset=["TiO2_wt_percent", "V_ppm"])   # rocks with both elements measured
ti_v = rocks_tv["TiO2_wt_percent"] * 5993 / rocks_tv["V_ppm"]
shervais = []
for r in ti_v:
    if r < 20:
        shervais.append("IAB")
    elif r < 50:
        shervais.append("MORB")
    else:
        shervais.append("OIB")
right = (np.array(shervais) == rocks_tv["setting"]).sum()
print("Shervais's lines:", right, "of", len(rocks_tv), "rocks right =", round(right / len(rocks_tv), 3))

plt.figure(figsize=(7, 4.5))
for setting, marker in [("IAB", "o"), ("MORB", "s"), ("OIB", "^")]:
    some = rocks_tv[rocks_tv["setting"] == setting]
    plt.scatter(some["TiO2_wt_percent"], some["V_ppm"], s=12, marker=marker, label=setting)
tio2 = np.arange(0, 5.01, 0.1)
for ratio in [20, 50]:
    plt.plot(tio2, tio2 * 5993 / ratio, color="black", lw=1)   # V = Ti / ratio
plt.text(1.2, 660, "Ti/V = 20")
plt.text(4.25, 450, "Ti/V = 50")
plt.xlim(0, 5)
plt.ylim(0, 700)
plt.xlabel("TiO2 (weight percent)")
plt.ylabel("V (parts per million)")
plt.title("Shervais's lines, drawn by hand in 1982")
plt.legend()
plt.show()

**0.792, from two lines a person drew.** Today's question is whether a computer can find
such lines, or such rules, itself.

**First, lock some rocks away.** The routine analysis of a rock lists nine major oxides:
silica, titanium, aluminium, calcium, magnesium, manganese, potassium, sodium and phosphorus. (Iron
is left out: more than half the rocks have no iron value, and the rest split it between FeO
and Fe2O3 in different ways.) The next cell keeps the 640 rocks with all nine and, as in the model-selection week,
splits them once: 448 training rocks to build and choose with, and 192
**sealed** rocks that nothing touches until the last Your turn. A score on rocks that helped choose
the model is not a test. Then it prints the commonest setting's share (`value_counts` lists the
commonest first) and checks the prediction above on the training rocks.

In [ ]:
oxides = ["SiO2_wt_percent", "TiO2_wt_percent", "Al2O3_wt_percent", "CaO_wt_percent", "MgO_wt_percent",
          "MnO_wt_percent", "K2O_wt_percent", "Na2O_wt_percent", "P2O5_wt_percent"]
rocks_ox = rocks.dropna(subset=oxides)   # rocks with all nine oxides measured
train, sealed = train_test_split(rocks_ox, test_size=0.3, random_state=0, stratify=rocks_ox["setting"])
print(len(train), "training rocks,", len(sealed), "sealed until the end")

counts = train["setting"].value_counts()
print("guess the commonest setting every time: right", round(counts.iloc[0] / len(train), 3), "of the time")
print(train.groupby("setting")[["TiO2_wt_percent", "K2O_wt_percent"]].median())

**The prediction holds.** The middle (median) arc rock holds 0.9 percent TiO2 and
0.51 percent K2O; the middle ridge rock 1.42 and
0.16; the middle ocean-island rock 2.78 and 0.94.
Titanium parts all three; potassium helps where titanium overlaps, ridge from arc at low titanium
and ridge from ocean island at high.

**The bar.** Write the dumbest rule you can, first. Any model that cannot beat it is decoration. Guessing the commonest setting is right 0.355 of the
time. Last week, when explosions were rare, the share right hid the misses, and you counted them
apart. Here the three settings are nearly equal in number (159, 151
and 138 training rocks), so the share right is a fair score; the confusion matrix
still shows where the misses fall.

**The training rocks, potassium against titanium:**

In [ ]:
two = ["TiO2_wt_percent", "K2O_wt_percent"]   # the two elements
plt.figure(figsize=(7, 4.5))
for setting, marker in [("IAB", "o"), ("MORB", "s"), ("OIB", "^")]:
    some = train[train["setting"] == setting]
    plt.scatter(some["TiO2_wt_percent"], some["K2O_wt_percent"], s=12, marker=marker, label=setting)
plt.xlabel("TiO2 (weight percent)")
plt.ylabel("K2O (weight percent)")
plt.title("The training rocks")
plt.legend()
plt.show()

### ✏️ Your turn 1

**How well do last week's straight lines sort the rocks on titanium and potassium?**

**Why:** last week's model, and last week's way of scoring it, are the bar every new model must clear.

**How logistic regression calls a rock, in one picture:**

<img src="https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data/images/week10/logistic_regression_mechanism.png" width="640">

*Each column (x1, x2) is multiplied by a weight (the weights ride on the arrows, not drawn) and
summed with a constant (b); the node marked ∫ stands for the S-shaped sigmoid, which turns the sum
into a probability, p. Where p is one half, a straight line runs between the two groups.
Figure: Shazam7734 (CC0), via Wikimedia Commons.*

With three settings, `LogisticRegression` makes one weighted sum for each setting, gives each rock
three probabilities that add up to 1, and calls it by the highest.

1. **`line_tk`**: fit it on the training rocks' two columns, `train[two]`, and their setting.
2. **`counts_tk`**: its confusion matrix, as last week: true settings, then the calls from
   `line_tk.predict(train[two])`. Three settings make three rows (the truth) and three columns (the
   calls); the given lines draw it as a grid.
3. **`score_tk`**: the share called right. The right calls run down the diagonal: add
   `counts_tk[0, 0]`, `counts_tk[1, 1]` and `counts_tk[2, 2]`, and divide by `len(train)`. The
   library's `.score` (given) does the same count, and from here on we use it. (How to score fairly,
   on rocks the model did not learn from, is section 2.)

**Use these names**, because the self-check looks for them: `line_tk`, `counts_tk` and `score_tk`.

In [ ]:
# ← your answer here: replace every ... with code
# train: the 448 training rocks; two: the column names of titanium and potassium
line_tk = LogisticRegression(max_iter=5000)                            # max_iter: more fitting steps than the default 100
line_tk.fit(..., ...)                                                  # the training rocks' two columns, and their setting
counts_tk = confusion_matrix(..., ..., labels=["IAB", "MORB", "OIB"])  # true settings, then the model's calls
plt.figure(figsize=(4.5, 4))
ConfusionMatrixDisplay(counts_tk, display_labels=["IAB", "MORB", "OIB"]).plot(ax=plt.gca(), cmap="Blues", colorbar=False)
plt.grid(False)
plt.title("Straight lines on Ti and K, training rocks")
plt.show()
score_tk = ...                                                         # the diagonal's three counts, added, over the number of training rocks
print("straight lines on titanium and potassium:", round(score_tk, 3), "of the training rocks right")
print("the library's .score says:", round(line_tk.score(train[two], train["setting"]), 3))

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert (np.array(counts_tk) == np.array([[135, 18, 6], [15, 128, 8], [1, 14, 123]])).all(), "confusion_matrix(train['setting'], line_tk.predict(train[two]), labels=[...])"
assert round(score_tk, 3) == 0.862, "the diagonal: counts_tk[0, 0] + counts_tk[1, 1] + counts_tk[2, 2], over len(train)"
print(f"✓ Your turn 1 — the straight lines get {score_tk:.3f} of the training rocks right")

**0.862, against 0.355 for a guess.** Read the matrix row by row, as last week's
recall: of 159 arc rocks it catches 135, of
151 ridge rocks 128, of 138 ocean-island
rocks 123. Of its 62 misses, 33 are arcs and ridges
mistaken for each other. And last week each event got one probability, of being an explosion; now each rock gets
three, one per setting, adding up to 1:

In [ ]:
one_rock = train.loc[[59], two]   # one training rock, picked by its index label
print(one_rock)
print("probabilities", line_tk.classes_, ":", line_tk.predict_proba(one_rock)[0].round(2))
print("they add up to", line_tk.predict_proba(one_rock)[0].sum().round(2))
print("called:", line_tk.predict(one_rock)[0])

**0.27, 0.71 and 0.01: the call is the largest,
MORB** (the rock is MORB). What did the model draw? The next cell
writes a small function that asks a fitted model to call every point of a fine grid of titanium and
potassium values, and colours the map by its calls: the model's **regions**. It is used again below.

In [ ]:
def draw_regions(model, title):
    """Colour a TiO2-K2O map by what a fitted model calls each point, and draw the training rocks."""
    ti_grid, k_grid = [], []
    for ti in np.arange(0, 4.51, 0.05):         # a grid of made-up rocks
        for k in np.arange(0, 2.51, 0.05):
            ti_grid.append(ti)
            k_grid.append(k)
    grid = pd.DataFrame({"TiO2_wt_percent": ti_grid, "K2O_wt_percent": k_grid})
    calls = model.predict(grid)                 # the model's call for each made-up rock
    plt.figure(figsize=(7, 4.5))
    for setting, colour in [("IAB", "tab:blue"), ("MORB", "tab:orange"), ("OIB", "tab:green")]:
        here = calls == setting
        plt.scatter(grid["TiO2_wt_percent"][here], grid["K2O_wt_percent"][here], s=8, marker="s", color=colour, alpha=0.3)
        some = train[train["setting"] == setting]
        plt.scatter(some["TiO2_wt_percent"], some["K2O_wt_percent"], s=12, color=colour, edgecolor="black", lw=0.3, label=setting)
    plt.xlabel("TiO2 (weight percent)")
    plt.ylabel("K2O (weight percent)")
    plt.title(title)
    plt.legend()
    plt.show()

draw_regions(line_tk, "Straight lines: three regions with straight edges")

**Three regions with straight edges.** Each edge is where two settings' probabilities tie. The model
can be written down, `line_tk.coef_` and `line_tk.intercept_` hold its numbers, but not as a rule a
geologist could apply with a hand lens and a table: a weighted sum for each setting, and the largest
wins.

**A geologist's kind of rule asks yes-or-no questions:** is the titanium above some value? A rule with
two cuts on titanium alone, `low` and `high`, would say: at or below `low`, an arc; up to `high`, a
ridge; above, an ocean island. The next cell writes that rule as a function and tries it on three
made-up rocks with cuts 1.0 and 2.0, where the answer is plain.

In [ ]:
def call(ti, low, high):
    """Call each rock from its TiO2: arc at or below low, ridge up to high, ocean island above."""
    guesses = []
    for t in ti:                     # one rock at a time
        if t <= low:
            guesses.append("IAB")    # little titanium: arc
        elif t <= high:
            guesses.append("MORB")   # middle: ridge
        else:
            guesses.append("OIB")    # much: ocean island
    return guesses

print(call([0.8, 1.5, 3.0], 1.0, 2.0))

**Which two cuts are best?** In the linear-regression week you searched a grid of slopes and
intercepts for the line with the smallest error. Here you search a grid of cut pairs for the rule
that calls the most training rocks right. Both cuts run from 0.5 to 4.0 in steps of 0.1, and `high`
must be above `low`: 630 pairs.

### ✏️ Your turn 2

**Which two titanium cuts call the most training rocks right?**

**Why:** a rule found by searching the data, not drawn by hand, is what a decision tree builds. Search
it yourself once, and the tree below will hold no mystery.

1. Start `best_right` at 0.
2. Loop over `low` in `cuts`, and inside it over `high` in `cuts`. Only when `high > low`:
   - `guesses`: `call` on the training rocks' `TiO2_wt_percent`, with `low` and `high`;
   - `right`: how many guesses match `train["setting"]` (as in the Shervais cell above);
   - if `right` is strictly more than `best_right`, keep all three: `best_right, best_low, best_high`.
3. Print the best cuts, the count and the share of training rocks right.

**Use these names**, because the self-check looks for them: `best_right`, `best_low`, `best_high`.

In [ ]:
# ← your answer here: replace every ... with code
cuts = np.arange(0.5, 4.01, 0.1).round(1)                        # 0.5, 0.6, ... 4.0; .round(1) trims tiny decimal errors
best_right = 0
for low in cuts:
    for high in cuts:
        if high > low:
            guesses = ...                                        # call on the training rocks' TiO2, with low and high
            right = ...                                          # how many guesses match train["setting"]
            if ...:                                              # better than the best so far?
                best_right, best_low, best_high = ..., ..., ...  # three names filled at once, in order
print("best cuts:", best_low, "and", best_high, "->", best_right, "of", len(train), "right =",
      round(best_right / len(train), 3))

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert best_right == 354, "count the guesses equal to train['setting'], and keep the largest"
assert (best_low, best_high) == (1.1, 2.1), "keep low and high together with best_right"
print(f"✓ Your turn 2 — cuts {best_low} and {best_high}, {best_right} rocks right")

**Cuts at 1.1 and 2.1 percent TiO2 call 354 of 448
training rocks right (0.79).** A rule anyone can apply, from one element. It is below the
straight lines (0.862), which also use potassium.

### A decision tree does the same search

**A decision tree.** A flowchart of yes/no questions, learned from the data instead of written by hand. At each step it tries every cut on every column it is
given and keeps the one that best parts the settings. It does not count rocks right, as you did; it
measures how mixed each side of the cut is, and picks the cut that leaves the two sides least mixed.
`max_depth=2` allows two questions in a row, the shape of your rule. The next cell fits one on
titanium alone, prints its questions with `export_text`, and scores your own rule at the tree's cuts.

In [ ]:
tree_ti = DecisionTreeClassifier(max_depth=2, random_state=0)
tree_ti.fit(train[["TiO2_wt_percent"]], train["setting"])     # double brackets: a table with one column
print(export_text(tree_ti, feature_names=["TiO2_wt_percent"]))
rule_at_tree = (np.array(call(train["TiO2_wt_percent"], 1.04, 2.13)) == train["setting"]).mean()   # cuts read from the printout
print("the tree:", round(tree_ti.score(train[["TiO2_wt_percent"]], train["setting"]), 3),
      "  your rule at the tree's cuts:", round(rule_at_tree, 3))

**How mixed is a side?** Take the tree's first cut, at 2.13 percent TiO2. Above it,
120 of the 134 training rocks are ocean-island basalt: a share of 0.9, close to
pure. At or below it, arcs and ridges are mixed: the commonest setting holds only 0.49 of the
314 rocks. A
side is "less mixed" the closer its commonest share is to 1 (the library uses a refined version of this
idea, called Gini impurity), and the tree keeps the cut whose two sides are least mixed.

**The tree asks "TiO2 <= 2.13?" first, then "TiO2 <= 1.04?" on the low side:
your two cuts, 1.1 and 2.1, moved by a few hundredths, and your rule at its
cuts scores exactly what it scores, 0.795.** (The printouts round its cuts to two decimals.) Its cuts beat your best (0.79)
only because they fall between your grid's steps of 0.1; the tree tries a cut between every pair of
neighbouring rocks. (It also asks a question on the high side
that changes nothing: both answers there say OIB. It splits wherever the mix of rocks changes, even
when the call does not.) The tree is your loop, with a different way to judge a cut. One tree is not
this week's model; it is the building block of one, the random forest of section 3.

**With more columns, more questions.** Given all nine oxides and three questions in a row, the tree
picks, at every step, the column and the cut that part the rocks best. Here is that tree. Each box
shows its question at the top; `samples` counts the training rocks that reach it, `value` splits them
by setting (IAB, MORB, OIB), and `class` is what the box would call them.

In [ ]:
tree3 = DecisionTreeClassifier(max_depth=3, random_state=0)
tree3.fit(train[oxides], train["setting"])
plt.figure(figsize=(14, 7))
plot_tree(tree3, feature_names=oxides, class_names=list(tree3.classes_), filled=False, impurity=False, fontsize=9)
plt.show()

**Its first question is still titanium: "TiO2_wt_percent <= 2.13?"** Of nine
oxides, the search chose the one Shervais chose. Follow one training rock down the drawing with the
numbers below, going left at each box whose answer is yes, then check with the tree itself.

In [ ]:
follow = train.loc[[173]]   # one training rock, picked by its index label
print(follow[oxides + ["setting"]].T)   # .T turns the row into a column, easier to read
print("the tree calls it:", tree3.predict(follow[oxides])[0])

## 2. Does the score depend on which rocks we hold out?

**A model is judged on rocks it did not learn from.** The sealed rocks are for the very end, so
while choosing we hold out some *training* rocks instead: fit on the rest, score on the held-out
ones. That is the validation set of the model-selection week. But which rocks to hold out? The next
cell holds out a different fifth of the training rocks 20 times and scores the same model each time:
the straight lines of Your turn 1, now on all nine oxides.

In [ ]:
cut_scores = []
for seed in range(20):
    fit_rows, held = train_test_split(train, test_size=0.2, random_state=seed, stratify=train["setting"])
    model = LogisticRegression(max_iter=5000)   # max_iter: allow more fitting steps than the default 100
    model.fit(fit_rows[oxides], fit_rows["setting"])
    cut_scores.append(model.score(held[oxides], held["setting"]))

plt.figure(figsize=(7, 3))
plt.scatter(range(20), cut_scores)
plt.xticks(range(0, 20, 2))
plt.xlabel("random split number (random_state)")
plt.ylabel("share right")
plt.title("One model, 20 validation splits")
plt.show()
print("lowest", round(np.min(cut_scores), 3), " highest", round(np.max(cut_scores), 3))

**The same model scores anywhere from 0.867 to 0.967, depending only on which
rocks were held out.** A difference of a few points between two models on one split can be
nothing but the split.

**Cross-validation.** Cut the rows into five parts; hold out each part in turn, train on the other four, and score on the one held out. Five scores, and every row tested once. The next cell labels the parts: rock 0 in
part 0, rock 1 in part 1, … rock 5 back in part 0. The split above already shuffled the rocks, so
each part is a fair mix.

In [ ]:
part = np.arange(len(train)) % 5   # %: the remainder after dividing by 5, so 0, 1, 2, 3, 4, 0, 1, ...
print(len(train), "training rocks; rocks in each part:")
print(pd.Series(part).value_counts().sort_index())

### ✏️ Your turn 3

**What does cross-validation say of the straight lines?**

**Why:** five held-out parts, every rock held out once, replace one lucky or unlucky split with an
average and a spread.

1. Make an empty list, **`line_scores`**.
2. For each part `held_part` in `range(5)`:
   - `fit_rows`: the training rocks **not** in that part, `train[part != held_part]`;
   - `held`: the rocks in that part;
   - fit a `LogisticRegression(max_iter=5000)` on `fit_rows`' oxides and setting, and append its score
     on `held` to `line_scores`.
3. Print the five scores, their mean, and their spread: the standard deviation, as in the probability
   week (`np.array(line_scores).std()`).

**Use this name**, because the self-check looks for it: `line_scores`.

In [ ]:
# ← your answer here: replace every ... with code
line_scores = []
for held_part in range(5):
    fit_rows = ...                                          # the training rocks not in part held_part
    held = ...                                              # the training rocks in part held_part
    line = LogisticRegression(max_iter=5000).fit(..., ...)  # fit_rows' oxides, and their setting
    line_scores.append(...)                                 # its score on held's oxides and setting
print("five parts:", np.round(line_scores, 3), " mean", round(np.mean(line_scores), 3),
      " spread", ...)                                       # the standard deviation of the five, rounded to 3

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert round(np.mean(line_scores), 3) == 0.906, "the mean of the five held-out scores"
assert [round(s, 3) for s in line_scores] == [0.933, 0.9, 0.9, 0.91, 0.888], "fit on train[part != held_part], score on train[part == held_part]"
print(f"✓ Your turn 3 — the lines' cross-validation mean is {np.mean(line_scores):.3f}")

**Five scores, 0.933 to 0.888, mean 0.906.** The library does
the same in one call. `cross_val_score(model, X, y, cv=folds)` fits and scores once per part and
returns the five scores; `PredefinedSplit(part)` tells it to use *your* five parts, so the two
answers can be compared number for number.

In [ ]:
X = train[oxides]                  # the nine oxides of every training rock
y = train["setting"]               # and its setting
folds = PredefinedSplit(part)      # your five parts
line_cv = np.mean(line_scores)     # the straight lines' mean: the bar to beat
print(np.round(cross_val_score(LogisticRegression(max_iter=5000), X, y, cv=folds), 3))

**The same five numbers.** From here on every model is judged by its cross-validation mean on these
five parts, and the straight lines' 0.906 is the bar.

## 3. Do a forest of trees, or a support vector machine, do better?

**Why a vote helps: a simple case.** Three voters, each right 80 percent of the time, each mistaken
independently of the others, like separate coin flips. The vote is right whenever at least two are right: all three right, probability
$0.8^3$, or exactly two, $3 \times 0.8^2 \times 0.2$ (three ways to pick the one who is wrong).

In [ ]:
all_three = 0.8 ** 3              # all three voters right
exactly_two = 3 * 0.8 ** 2 * 0.2  # two right, one wrong, in three ways
print("the vote is right", round(all_three + exactly_two, 3), "of the time; one voter alone 0.8")

**0.896, better than any one voter.** The gain lasts only while the voters' mistakes differ:
three copies of one tree vote the same way every time. To grow different trees from the same rocks,
give each tree a different **bootstrap sample**, the resampling of the uncertainty week: as many rocks
as there are, drawn at random *with replacement*, so each tree sees some rocks twice and others not
at all.

**How a random forest calls a rock, in one picture:**

<img src="https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data/images/week10/random_forest_mechanism.png" width="640">

*Many decision trees each give a call; the forest's call is the majority vote. (The figure feeds the
same data to every tree; in a random forest each tree gets its own bootstrap sample, as in Your turn 4.) Figure: TseKiChun (CC BY-SA 4.0), via Wikimedia Commons.*

### ✏️ Your turn 4

**Do 25 trees voting beat one tree?**

**Why:** this is what a random forest does inside. Build it by hand once, on one held-out part, and
the library's forest will hold no mystery.

`fit_rows` and `held` are part 0's split, as in Your turn 3 (given).

1. **Grow 25 trees.** Each pass: `sample`, a bootstrap sample of `fit_rows`
   (`.sample(len(fit_rows), replace=True, random_state=rng)`); a tree fitted on it; `guess`, its calls
   for `held`. The loop keeps every `guess` in `votes`, and its share right in **`one_tree`**.
2. **Count the vote.** `table` has one row per tree and one column per rock. For each rock,
   `table[rock].value_counts()` counts the trees behind each call, most first; the rock's call is the
   first of them, `.index[0]`. (On a tie, pandas puts one of the tied calls first; here that happens
   for 1 of the 90 rocks.)
3. **`vote_score`**: the share of `vote` equal to `held["setting"].values`.

**Use these names**, because the self-check looks for them: `one_tree` and `vote_score`.

In [ ]:
# ← your answer here: replace every ... with code
# train: the 448 training rocks; part: which of the five parts each is in
fit_rows = train[part != 0]
held = train[part == 0]
rng = np.random.default_rng(0)                                   # one random generator: each pass of the loop draws a new sample
votes = []
one_tree = []
for i in range(25):
    sample = ...                                                 # a bootstrap sample of fit_rows, as many rocks, with replacement, random_state=rng
    tree = DecisionTreeClassifier(random_state=0).fit(..., ...)  # the sample's oxides, and setting
    guess = ...                                                  # the tree's calls for held's oxides
    votes.append(guess)
    one_tree.append(...)                                         # the share of guess equal to held["setting"]
table = pd.DataFrame(votes)                                      # one row per tree, one column per rock
vote = []
for rock in table.columns:
    counts_rock = ...                                            # how many trees gave each call for this rock
    vote.append(...)                                             # the commonest call
vote_score = ...                                                 # the share of np.array(vote) equal to held["setting"].values
print("one tree, on average:", round(np.mean(one_tree), 3), "  lowest", round(np.min(one_tree), 3),
      " highest", round(np.max(one_tree), 3))
print("the vote of 25:", round(vote_score, 3))

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert round(np.min(one_tree), 3) == 0.833, "each tree's share of held-out rocks right"
assert round(np.mean(one_tree), 3) == 0.894, "each tree on fit_rows.sample(len(fit_rows), replace=True, random_state=rng)"
assert round(vote_score, 3) == 0.933, "vote.append(table[rock].value_counts().index[0]); (np.array(vote) == held['setting'].values).mean()"
print(f"✓ Your turn 4 — one tree {np.mean(one_tree):.3f} on average, the vote {vote_score:.3f}")

**One tree gets 0.894 of the 90 held-out rocks right on average (from
0.833 to 0.944, depending on its sample); their vote gets
0.933.** The vote beats the average tree, and 21 of the 25 single trees.

**A random forest.** Ask many slightly different trees and take a vote. `RandomForestClassifier` does what you just did, with one
more twist: at each question a tree may look at only some of the columns, so the trees differ even
more. The next cell scores 200 trees by cross-validation on your five parts, then draws the
forest's **feature importance**, which oxides it leaned on. How much of a forest's decisions each column carried — the model's own account of what it used. Each
question a tree asks makes its two sides less mixed; that gain is credited to the question's column,
added up over all 200 trees, and scaled so the credits sum to 1.

In [ ]:
forest_scores = cross_val_score(RandomForestClassifier(n_estimators=200, random_state=0), X, y, cv=folds)
print("forest:", np.round(forest_scores, 3), " mean", round(forest_scores.mean(), 3))

forest = RandomForestClassifier(n_estimators=200, random_state=0).fit(X, y)
importance = pd.Series(forest.feature_importances_, index=oxides).sort_values()
plt.figure(figsize=(7, 4))
plt.barh(importance.index, importance.values)
plt.xlabel("share of the credit")
plt.title("Which oxides the forest used")
plt.show()

**The forest scores 0.924, above the straight lines (0.906) on
3 of the 5 parts.** It leans most on titanium (0.321), then far less on
K2O (0.128) and SiO2 (0.119). Importance is the
model's own account of what it used, not a law of nature: two columns that carry the same
information share the credit between them.

### Support vector machine (SVM): the widest gap, and why units matter

**The widest gap.** Of all the lines separating the two groups, take the one leaving the widest gap.

<img src="https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data/images/week10/svm_widest_gap.png" width="380">

*Two groups (blue, green) and the line that parts them (red) with the widest gap (yellow band). The
points on the edge of the band, circled, are the **support vectors**: they alone fix the line, which
gives the method its name. Figure: Larhmam (CC BY-SA 4.0), via Wikimedia Commons.*

### ✏️ Your turn 5

**Where is the widest gap between two groups of pebbles?**

**Why:** this is what an SVM looks for. Find it by hand once, on a ruler, and the library's answer
will hold no mystery.

Six pebbles: small ones at 1, 2 and 3 cm, large ones at 7, 8 and 9. Any cut between 3 and 7 separates
them; its **gap** is the distance from the cut to the nearest pebble.

1. **`gaps`**: for each cut in `cuts`, its gap: `np.min(np.abs(sizes - cut))`.
2. **`best_cut`**: the cut with the widest gap; `np.argmax` gives its place.

**Use these names**, because the self-check looks for them: `gaps` and `best_cut`.

In [ ]:
# ← your answer here: replace every ... with code
sizes = np.array([1, 2, 3, 7, 8, 9])
cuts = np.arange(3, 7.01, 0.5)
gaps = []
for cut in cuts:
    gaps.append(...)  # the distance from cut to the nearest pebble
best_cut = ...        # the cut with the widest gap
print("gaps:", gaps)
print("the widest gap:", best_cut, "cm, gap", np.max(gaps))

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert len(gaps) == 9 and np.max(gaps) == 2.0, "gaps.append(np.min(np.abs(sizes - cut)))"
assert best_cut == 5.0, "cuts[np.argmax(gaps)]"
print(f"✓ Your turn 5 — the widest gap at {best_cut} cm")

`SVC` (support vector classifier, the widest-gap model) is asked next for a straight cut,
`kernel="linear"`; its cut is where $w \times \text{size} + b = 0$, at size $= -b / w$.

In [ ]:
gap = SVC(kernel="linear").fit(pd.DataFrame({"size": sizes}), ["small"] * 3 + ["large"] * 3)
print("SVC's cut:", round(-gap.intercept_[0] / gap.coef_[0][0], 2), "cm")

**Both put the cut at 5.0 cm, 2 cm from each group.** Real rocks are harder in two ways. Their clouds
overlap, so no cut is clean: the widest gap then trades a few rocks on the wrong side for a wider
gap. And there are three settings: `SVC` draws a gap between each pair (arc–ridge, arc–island,
ridge–island), each pair votes for one of its two, and the setting with the most votes wins.
Called as plain `SVC()`, as from here on, it may also bend its boundary into a curve to follow the
clouds: it then measures how close a rock is to the support vectors rather than which side of a
straight line it is on. The idea, the widest gap kept by a few rocks on its edge, is the same.

**A gap is a distance, and a distance adds up the differences in each column in that column's own
units.** The oxides' spreads (standard deviations) are very different:

In [ ]:
train[oxides].std().round(2)

**Magnesium, aluminium and silica vary by about 2 percent from rock to rock; manganese by
0.03, phosphorus by 0.22, potassium by
0.55.** In raw units a distance across the columns is decided by the big
oxides, and the small ones barely count. **Scaling** fixes this: from each column subtract its mean
and divide by its standard deviation, $z = (x - \text{mean}) / \text{std}$, so every column has
mean 0 and spread 1. A simple case: 2, 4 and 6 have mean 4 and standard deviation 2 (pandas' `.std()`), so they become
−1, 0 and 1. (Pandas' `.std()` divides by n − 1; NumPy's `np.std` and `array.std()`, used in Your
turn 3, divide by n. For hundreds of rocks the two differ by a fraction of a percent.) The mean and standard deviation must come from the rocks the model learns from only;
the held-out rocks are scaled with those same two numbers, or their own values leak into the model.

### ✏️ Your turn 6

**Does the SVM need its columns scaled?**

**Why:** a gap measured in mixed units is decided by the oxides that vary most. Scaling by hand once
shows what `StandardScaler` does.

`fit_rows` and `held` are still part 0's, from Your turn 4.

`raw_score`, an `SVC()` on the raw oxides, is given.

1. `mean` and `std`: `fit_rows[oxides].mean()` and `.std()`.
2. `fit_scaled` and `held_scaled`: both tables' oxides, minus `mean`, divided by `std`.
3. **`scaled_score`** (given) fits and scores on the scaled tables.

**Use these names**, because the self-check looks for them: `raw_score` and `scaled_score`.

In [ ]:
# ← your answer here: replace every ... with code
raw_score = SVC().fit(fit_rows[oxides], fit_rows["setting"]).score(held[oxides], held["setting"])
mean = ...         # each oxide's mean over fit_rows
std = ...          # each oxide's standard deviation over fit_rows
fit_scaled = ...   # fit_rows' oxides, minus mean, divided by std
held_scaled = ...  # held's oxides, scaled with the same mean and std
scaled_score = SVC().fit(fit_scaled, fit_rows["setting"]).score(held_scaled, held["setting"])
print("SVM on part 0, raw columns:", round(raw_score, 3), "  scaled:", round(scaled_score, 3))

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert round(raw_score, 3) == 0.789, "SVC() fitted on fit_rows[oxides], scored on held[oxides]"
assert round(scaled_score, 3) == 0.933, "scale both with fit_rows' mean and std"
print(f"✓ Your turn 6 — raw {raw_score:.3f}, scaled {scaled_score:.3f}")

**From 0.789 to 0.933 on part 0, by changing nothing but the units.**
`StandardScaler` does the scaling, and `make_pipeline(StandardScaler(), SVC())` joins it to the model
so that, inside `cross_val_score`, each part is scaled by what the other four learned, as you just did
by hand. Over all five parts:

In [ ]:
svm_raw_scores = cross_val_score(SVC(), X, y, cv=folds)
svm_scores = cross_val_score(make_pipeline(StandardScaler(), SVC()), X, y, cv=folds)
print("SVM, raw:", round(svm_raw_scores.mean(), 3), "  scaled:", round(svm_scores.mean(), 3))
print("scaled, part by part:", np.round(svm_scores, 3))

**0.781 raw, 0.922 scaled, and on part 0 the pipeline scores 0.933,
your number.** (`StandardScaler` divides by a standard deviation computed with n rather than n − 1,
as pandas' `.std()` does; scaling every column by the same small factor changes no call.) Every model that measures a distance across columns
needs this step, and you will meet it again in the next two weeks.

## 4. Which model wins when the sealed rocks are opened?

Every choice so far was made on the training rocks' five parts:

In [ ]:
names = ["straight lines", "forest", "SVM, scaled"]
models = [LogisticRegression(max_iter=5000),                       # the three, unfitted, in the same order
          RandomForestClassifier(n_estimators=200, random_state=0),
          make_pipeline(StandardScaler(), SVC())]
scores = [line_cv, forest_scores.mean(), svm_scores.mean()]
spreads = [np.array(line_scores).std(), forest_scores.std(), svm_scores.std()]
for name, score, spread in zip(names, scores, spreads):   # zip: walk the three lists side by side
    print(f"{name:15s} mean {score:.3f}   spread {spread:.3f}")
print("parts where the forest beats the scaled SVM:", (forest_scores > svm_scores).sum(), "of 5")

**The forest leads the scaled SVM by 2 rocks in a
thousand, far less than either's spread (0.031 and 0.023), and beats it on only
3 of the 5 parts.** The two are tied as far as these rocks can tell. A rule must still
pick one, before the look, and the usual rule is the highest mean. Now, and only now, open the sealed
rocks. You get one look: whatever it says is the answer, because a model tuned again after a look at
the test is no longer tested by it.

### ✏️ Your turn 7

**Which model is chosen, and how does it do on the sealed rocks?**

**Why:** the score on rocks that took no part in any choice is the only honest estimate of how the
model will do on the next rock.

1. **`winner`**: the name with the highest mean; `np.argmax` gives its place, as when you picked the
   best degree in the model-selection week.
2. **`chosen`**: the model in the same place of `models`, fitted on all the training rocks, `X` and `y`.
3. **`sealed_score`**: its score on the sealed rocks' oxides and setting. Print it beside the guess's
   share on the sealed rocks.
4. **`sealed_counts`**: its confusion matrix on the sealed rocks, as in Your turn 1 (the drawing is
   given).

**Use these names**, because the self-check looks for them: `winner`, `chosen` and `sealed_score`.

In [ ]:
# ← your answer here: replace every ... with code
# X, y: the training rocks' oxides and settings; sealed: the locked rocks; counts: training settings, commonest first
# names, models, scores: the three candidates, in the same order
winner = ...          # the name with the highest mean
print("chosen on the five parts:", winner)
chosen = ...          # the model in the same place of models
chosen.fit(..., ...)  # all the training rocks
sealed_score = ...    # its score on the sealed rocks' oxides and setting
print(winner, "on the sealed rocks:", round(sealed_score, 3),
      "  guessing:", round((sealed["setting"] == counts.index[0]).mean(), 3))
sealed_counts = ...   # the confusion matrix on the sealed rocks, labels IAB, MORB, OIB
plt.figure(figsize=(4.5, 4))
ConfusionMatrixDisplay(sealed_counts, display_labels=["IAB", "MORB", "OIB"]).plot(ax=plt.gca(), cmap="Blues", colorbar=False)
plt.grid(False)
plt.title("The chosen model on the sealed rocks")
plt.show()

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert winner == "forest", "names[np.argmax(scores)]"
assert round(sealed_score, 3) == 0.948, "fit on X, y; score on sealed[oxides], sealed['setting']"
print(f"✓ Your turn 7 — {sealed_score:.3f} of the sealed rocks right")

**182 of 192 sealed rocks right (0.948), against
0.354 for a guess.** Of the
10 misses, 5 are arcs and ridges mistaken for each other
(2 arc rocks called ridge, 3 ridge rocks called arc), the two settings whose
titanium overlaps most.

**Two more numbers, read only now that the choice is made.** Had the scaled SVM been chosen, it
would have scored 0.953 on the same rocks: the tie on the five parts was real, and either
choice would have served. Last week's straight lines, on the same nine oxides, score 0.911.
And your two titanium cuts, fixed before the look, call 0.849 of the sealed rocks right:

In [ ]:
lines_sealed = models[0].fit(X, y).score(sealed[oxides], sealed["setting"])
print("last week's straight lines on the sealed rocks:", round(lines_sealed, 3))
rule_sealed = (np.array(call(sealed["TiO2_wt_percent"], best_low, best_high)) == sealed["setting"]).mean()
print("your two titanium cuts on the sealed rocks:", round(rule_sealed, 3))

**One element and two cuts, 0.849; nine oxides and a forest, 0.948.** The
extra chemistry and the vote are worth about 10 rocks in
a hundred here. (Split the same rocks 20 ways and the gap runs from 9 to 17, typically
14; the forest's own score runs from 0.9 to 0.95.)

## The question, answered

**From its chemistry, nearly always, for rocks like these: a forest on nine oxides names the setting
of 182 of 192 rocks it never saw (0.948).** Titanium carries
the most: one element with two cuts found by your loop already got 0.79 of the training
rocks right. A vote of trees was a little better than last week's straight lines, on
3 of the 5 parts, and the scaled SVM as good. And the
score was honest because the rocks that gave it took no part in choosing the model. One caution:
these rocks all come from Vermeesch's compilation, and rocks from one volcano can sit on both sides of
a split. A basalt from somewhere nobody sampled may be harder. That is the homework's last question.

### Three ways to call a rock, side by side

| | Straight lines (logistic regression) | Random forest | Support vector machine (SVM) |
|---|---|---|---|
| **How it calls a rock** | one weighted sum of the columns per setting, turned into a probability; the highest wins | a vote of many decision trees, each a chain of yes/no cuts grown on a bootstrap sample and shown only some columns at each question | the side of a boundary kept as far as it can be from the settings on either side: the widest gap |
| **What you choose** | nothing new this week | how many trees (`n_estimators`; 200, fixed here) | scale the columns first |
| **Its strength** | simple, and rarely beaten by much when the columns are few | the best mean here, by a hair | as good as the forest, once scaled |
| **Its weakness** | its boundaries are straight | it cannot be read as one flowchart, as one tree can; importance is only its own account | in raw units the big columns decide the gap |
| **Nine oxides, five parts** | 0.906 | 0.924 | 0.781 raw, 0.922 scaled |

**Which to use:** start with the straight lines, the bar to beat. Use a forest when the score matters
most and there are many columns; an SVM only on scaled columns. And whichever you pick, choose it on
held-out parts and test it once.

## Week 10 summary

### The science

**Where was this basalt made?**  
A basalt carries the mark of where it melted: titanium is poorest in arc basalt and richest in ocean-island basalt, and potassium, carried up by water from the sinking plate, is higher in arc basalt than in ridge basalt. Last week's straight lines on the two call 0.86 of the training rocks right; two titanium cuts found by a loop, a rule anyone can apply, 0.79. On the sealed rocks, straight lines on nine oxides call 0.91 right and a forest 0.95 (0.90 to 0.95 on other splits). Three ways to call a rock, each with one picture. Logistic regression (straight lines) weighs the columns, sums them and turns each sum into a probability; it is rarely beaten by much when the columns are few. A random forest is a vote of decision trees, each a chain of yes/no cuts grown on a bootstrap sample: the best mean here, by a hair, but it cannot be read as one flowchart. A support vector machine keeps its boundary as far as it can from both sides, the widest gap, and works well only on columns brought to one scale, learned from the training rows. One validation split can move a score by ten points; cross-validation holds out each fifth in turn. Choose on the training rocks' five parts, open the test rocks once, and know that rocks from another compilation can score lower (here 0.87 against 0.95).

- **Three homes of basalt.** Basalt is made in three places, where the mantle melts: where two plates pull apart (a mid-ocean ridge), where one plate sinks under another (an island arc), and over a hot plume, usually far from any plate edge (an ocean island).
- **Major oxide, trace element.** A rock analysis gives the common elements as oxides, in percent of the rock's weight (SiO2, TiO2), and the rare ones in parts per million (V, Sr).
- **Feature importance.** How much of a forest's decisions each column carried — the model's own account of what it used.
- **Cross-validation.** Cut the rows into five parts; hold out each part in turn, train on the other four, and score on the one held out. Five scores, and every row tested once.
- **Decision tree.** A flowchart of yes/no questions, learned from the data instead of written by hand.
- **Random forest.** Ask many slightly different trees and take a vote.
- **SVM.** Of all the lines separating the two groups, take the one leaving the widest gap.

### The code

| Function | What it does |
|---|---|
| **Python** | |
| `export_text(tree, feature_names=columns)` | print a fitted tree's questions as text |
| `ConfusionMatrixDisplay(counts, display_labels=names).plot()` | draw a confusion matrix as a grid shaded by count, true labels down the side, calls along the bottom |
| `make_pipeline(StandardScaler(), SVC())` | one model that scales the columns and then fits; inside cross-validation each part learns its own scaling |
| `cross_val_score(model, X, y, cv=5)` | hold out each of five parts in turn, fit on the other four, and return the five scores; cv=PredefinedSplit(part) makes the parts your own, as in class |
| `def name(a, b):` | write the recipe once and give it a name; the names in brackets are what it needs |
| `return value` | hand one value back to whoever called the function |
| `for x in things:` | do the same thing once for each item |
| `range(n)` | the numbers 0 to n-1, to count with |
| `if / elif / else` | choose what to do, based on a comparison |
| `list.append(x)` | add one item to the end of a list |
| `len(table)` | how many rows |
| `round(x, 1)` | trim a long decimal — 9.1 rather than sixteen digits |
| **NumPy** | |
| `np.array(list)` | the container that does arithmetic to every number at once |
| `np.argmax(values) / np.argmin(values)` | the position of the largest / smallest value, to read the same position out of another array or list |
| `np.max(values) / np.min(values)` | the largest / smallest value of an array or a list |
| `array.mean()` | the average; on a True/False array, the fraction that are True |
| `array.std()` | the standard deviation: how far a value usually is from the average |
| `np.random.default_rng(seed)` | a random-number generator you can reproduce — the same seed gives the same draws |
| **pandas** | |
| `table.sample(n, replace=True, random_state=rng)` | a new table of n rows drawn at random from this one, each row put back so it can come again |
| **Matplotlib** | |
| `plt.figure(figsize=(w, h))` | start a new figure, w inches wide and h tall — the first line of every plot cell |
| `plt.xlabel / ylabel / title / show` | label it, then draw it |
| `plt.legend()` | name the things on a plot when there is more than one |
| `plt.scatter(x, y)` | a dot per pair of values |
| **scikit-learn** | |
| `DecisionTreeClassifier(max_depth=3, random_state=0)` | learn a flowchart of yes/no questions from the data, at most max_depth questions deep |
| `RandomForestClassifier(n_estimators=200, random_state=0)` | many slightly different trees, voting |
| `SVC()` | the widest-gap classifier; give it scaled columns or the gap means nothing |
| `StandardScaler()` | put every column on the same footing, so a distance across them means something |
| `forest.feature_importances_` | one number per column: how much of the forest's decisions it carried |
| `model.fit(X_train, y_train)` | learn from the training rows |
| `LogisticRegression()` | draw the straight line that best separates two labelled groups; with three labels, one weighted sum per label, the edges where two sums tie |
| `model.predict(x)` | the fitted model's answer for each row of x: a number from a line, a label from a classifier |
| `confusion_matrix(y, pred)` | count right and wrong calls: true settings down the side, calls across, the right calls on the diagonal |
| `train_test_split(table, test_size=0.3, random_state=n, stratify=column)` | cut the rows into a training part and a held-out part, keeping the share of each label |

## Homework

**Three parts, each a step of class with less given.** Parts 1 and 2 ask which two elements a geologist
should measure, if only two: titanium and vanadium, Shervais's pair, or titanium and strontium.
Strontium is the best measured trace element in the file (blank for 49 of 756 rocks), and arc melts
are richer in it than ridge melts: water from the sinking plate carries it into the melt. It has a weakness: sea water
moves strontium in and out of a basalt after it erupts, while titanium and vanadium stay put. Part 3 asks whether the class's
forest works on rocks from a different compilation.

The first cell remakes what the parts use, with the rocks in a shuffled order and the five parts
labelled as in class. After a restart, run the Setup cell at the top first, then this one.

In [ ]:
# ── Checkpoint ── run this if you restarted the kernel or fell behind ──
rocks = pd.read_csv(CACHE + "/week10_vermeesch_basalts.csv")   # the basalts again, read and fixed as in Setup
rocks = rocks.rename(columns={"affinity": "setting", "P2O5(wt%)": "P2O5_wt_percent"})
rocks.loc[rocks["MnO_wt_percent"] > 5, "MnO_wt_percent"] = np.nan
oxides = ["SiO2_wt_percent", "TiO2_wt_percent", "Al2O3_wt_percent", "CaO_wt_percent", "MgO_wt_percent",
          "MnO_wt_percent", "K2O_wt_percent", "Na2O_wt_percent", "P2O5_wt_percent"]
rocks_ts = rocks.dropna(subset=["TiO2_wt_percent", "Sr_ppm"])   # titanium and strontium
rocks_ts = rocks_ts.sample(len(rocks_ts), random_state=0)       # shuffled once
X_ts, y_ts = rocks_ts[["TiO2_wt_percent", "Sr_ppm"]], rocks_ts["setting"]
part_ts = np.arange(len(rocks_ts)) % 5
rocks_tv = rocks.dropna(subset=["TiO2_wt_percent", "V_ppm"])    # titanium and vanadium
rocks_tv = rocks_tv.sample(len(rocks_tv), random_state=0)
X_tv, y_tv = rocks_tv[["TiO2_wt_percent", "V_ppm"]], rocks_tv["setting"]
part_tv = np.arange(len(rocks_tv)) % 5
rocks_ox = rocks.dropna(subset=oxides)                          # all nine oxides, for part 3
print(len(rocks_ts), "rocks with TiO2 and Sr;", len(rocks_tv), "with TiO2 and V")

### ✏️ Your turn 8 (homework part 1)

**Make cross-validation a function. Do titanium and strontium beat titanium and vanadium?**

**Why:** in class you wrote the five-part loop once. A function makes it a tool you can point at any
model and any table.

1. Write **`cv_mean(model, table, labels, parts)`**: the loop of Your turn 3, for any model, any table
   of columns, any labels and any part labels. It returns the mean of the five scores.
   (`table[parts != held_part]` and `labels[parts != held_part]` pick the rows outside that part.)
2. **`line_ts`**: `cv_mean` of a `LogisticRegression(max_iter=5000)` on `X_ts`, `y_ts`, `part_ts`.
   **`line_tv`**: the same on `X_tv`, `y_tv`, `part_tv`. Print both.

**Use these names**, because the self-check looks for them: `cv_mean`, `line_ts` and `line_tv`.

In [ ]:
# ← your answer here: replace every ... with code
def cv_mean(model, table, labels, parts):
    """The mean of the five held-out scores, each part held out once."""
    ...

line_ts = ...  # the straight lines on titanium and strontium
line_tv = ...  # the straight lines on titanium and vanadium
print("lines on Ti and Sr:", round(line_ts, 3), "  on Ti and V:", round(line_tv, 3))

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert round(cv_mean(LogisticRegression(max_iter=5000), X_ts, y_ts, part_ts), 3) == 0.921, "for held_part in range(5): fit on table[parts != held_part], score on table[parts == held_part]; return the mean"
assert round(line_ts, 3) == 0.921 and round(line_tv, 3) == 0.852, "cv_mean on X_ts, y_ts, part_ts and on X_tv, y_tv, part_tv"
print(f"✓ Your turn 8 — Ti and Sr {line_ts:.3f}, Ti and V {line_tv:.3f}")

**What the numbers mean.** Which pair scores higher, and by how much? Sea water moves strontium in and out of a basalt after it
erupts; titanium and vanadium stay put. Why might a geologist still prefer titanium and vanadium?

### ✏️ Your turn 9 (homework part 2)

**On titanium and strontium, do the forest and the SVM beat the straight lines?**

**Why:** on nine oxides the forest and the scaled SVM were a little ahead of the lines. Is that true
of any data, or does it need many columns? And strontium is counted in hundreds of parts per million,
titanium in a few percent: does the SVM notice?

With your `cv_mean` on `X_ts`, `y_ts`, `part_ts`:

1. **`forest_ts`**: a `RandomForestClassifier(n_estimators=200, random_state=0)`.
2. **`svm_ts_raw`**: an `SVC()` on the raw columns.
3. **`svm_ts_scaled`**: the pipeline that scales and then fits, as in class.
4. Print the three beside `line_ts` (given), and each column's standard deviation, `X_ts.std()`.

**Use these names**, because the self-check looks for them: `forest_ts`, `svm_ts_raw` and `svm_ts_scaled`.

In [ ]:
# ← your answer here: replace every ... with code
forest_ts = ...      # the forest
svm_ts_raw = ...     # the SVM on the raw columns
svm_ts_scaled = ...  # scale, then the SVM
print("lines:", round(line_ts, 3), "  forest:", round(forest_ts, 3),
      "  SVM raw:", round(svm_ts_raw, 3), "  SVM scaled:", round(svm_ts_scaled, 3))
print(...)           # each column's standard deviation: X_ts's .std(), rounded to 2 places

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert round(forest_ts, 3) == 0.906, "cv_mean(RandomForestClassifier(n_estimators=200, random_state=0), X_ts, y_ts, part_ts)"
assert round(svm_ts_raw, 3) == 0.654, "cv_mean(SVC(), X_ts, y_ts, part_ts)"
assert round(svm_ts_scaled, 3) == 0.924, "cv_mean(make_pipeline(StandardScaler(), SVC()), X_ts, y_ts, part_ts)"
print(f"✓ Your turn 9 — forest {forest_ts:.3f}, SVM raw {svm_ts_raw:.3f}, scaled {svm_ts_scaled:.3f}")

**What the numbers mean.** Which wins on two elements: the lines, the forest or the SVM? How much does scaling change the SVM's
score? Look at the two standard deviations: in raw units, which element decides the size of the gap?

### Rocks from somewhere else

The sealed rocks came from Vermeesch's own compilation, and rocks from one volcano can sit on both
sides of a split. In 2022 the geologist Luc Doucet and colleagues compiled a much larger set of
basalts of known setting, from two online databases (EarthChem and GEOROC), kept to rocks in the basalt silica range
(43 to 53 percent silica, sodium plus potassium oxides below 5 percent; the rocks read here run from
45 to 53). Their own model, judged by
cross-validation, was a vote of a tree, a widest-gap line and a forest: the methods of today. (It
used 40 measures built from trace elements on 1,012 well-studied rocks and scored 96.6 percent; ours
uses nine oxides, so the two scores are not comparable.) The next cell reads 2400 of their
rocks, 800 per setting, with the same nine oxides and where each came from: a place, or for every
ridge rock, PetDB, a database of ridge rocks from around the world.

In [ ]:
# Doucet et al. (2022), Earth-Science Reviews 233, 104192, Supplementary Table S1:
# 800 rocks per setting, with all nine oxides (ARC-O, their oceanic arcs, renamed IAB).
doucet = pd.read_csv(CACHE + "/week10_doucet2022_basalts.csv")
print(list(doucet.columns))   # the same nine oxide names as class
print(doucet.groupby("setting")["location"].value_counts())

### ✏️ Your turn 10 (homework part 3)

**Would the class's forest work on rocks from somewhere else?**

**Why:** a test on rocks from the same compilation says how a model does on rocks like those. Doucet's
rocks were gathered by other people, by other rules, and every arc rock is labelled Izu-Bonin.

Fit a forest like the class's on **all** 640 Vermeesch rocks with nine oxides (`rocks_ox`, from
the checkpoint: the training and sealed rocks together, because the test is now Doucet's rocks), and
score it once on Doucet's.

1. **`forest_all`**: a `RandomForestClassifier(n_estimators=200, random_state=0)` fitted on
   `rocks_ox`'s oxides and setting.
2. **`doucet_score`**: its score on `doucet`'s oxides and setting. Print it.
3. **`doucet_counts`**: its confusion matrix on Doucet's rocks (the drawing is given).

**Use these names**, because the self-check looks for them: `forest_all` and `doucet_score`.

In [ ]:
# ← your answer here: replace every ... with code
# rocks_ox: the 640 Vermeesch rocks with all nine oxides; doucet: Doucet's 2,400 rocks
forest_all = RandomForestClassifier(n_estimators=200, random_state=0)
...                  # fit it on rocks_ox's oxides and setting
doucet_score = ...   # its score on Doucet's oxides and setting
print("the forest on Doucet's rocks:", round(doucet_score, 3))
doucet_counts = ...  # its confusion matrix on Doucet's rocks, labels IAB, MORB, OIB
plt.figure(figsize=(4.5, 4))
ConfusionMatrixDisplay(doucet_counts, display_labels=["IAB", "MORB", "OIB"]).plot(ax=plt.gca(), cmap="Blues", colorbar=False)
plt.grid(False)
plt.title("The forest on Doucet's rocks")
plt.show()

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert round(doucet_score, 3) == 0.869, "fit on rocks_ox[oxides], rocks_ox['setting']; score on doucet[oxides], doucet['setting']"
print(f"✓ Your turn 10 — {doucet_score:.3f} of Doucet's rocks right")

**What the numbers mean.** How far does the score fall from the sealed rocks'? Where do Doucet's rocks of each setting come from?
What does that say about the score you found in class?